<a href="https://colab.research.google.com/github/recy214/Programacion-de-Sistemas-Base-I/blob/main/Martinez_Juan_PSSB_I_analizador_lexico_colab_U3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GUÍA PRÁCTICA Y EVALUACIÓN ASÍNCRONA: ANALIZADORES LÉXICOS EN PYTHON Y JAVA
**Tema: Generación Automática de Analizadores Léxicos**  
**Asignatura:** Programación de Sistemas de Bas 1  

---  
## 🎯 Objetivos de Aprendizaje
1. **Comprender** la generación automática de analizadores léxicos a partir de especificaciones formales y expresiones regulares.
2. **Implementar y extender** escáneres léxicos en dos entornos tecnológicos representativos: **PLY (`ply.lex`)** en Python y **JFlex** en Java.
3. **Analizar** los mecanismos de disambiguación léxica (palabras reservadas vs. identificadores, coincidencia más larga) y el manejo de atributos/posiciones.
4. **Evaluar** estructuralmente las diferencias entre enfoques interpretados/dinámicos (PLY) y compilados/estáticos (JFlex).

---  
## 🐍 PARTE 1: Analizador Léxico en Python utilizando PLY (`ply.lex`)

### 1.1 Código Base e Instalación

In [19]:
# Instalación de la librería PLY en el entorno de Google Colab
!pip install ply > /dev/null

In [20]:
%%writefile lexer_ply.py

import ply.lex as lex

class PythonLexerPLY:
    """Especificación e implementación de un Analizador Léxico en Python usando PLY."""

    # 1. LISTA OBLIGATORIA DE TOKENS
    tokens = (
        'KEYWORD_IF',
        'KEYWORD_ELSE',
        'KEYWORD_WHILE',
        'KEYWORD_INT',
        'KEYWORD_FLOAT',
        'IDENTIFIER',
        'INTEGER_LITERAL',
        'FLOAT_LITERAL',
        'ASSIGN',
        'PLUS',
        'MINUS',
        'TIMES',
        'DIVIDE',
        'SEMICOLON',
        'LPAREN',
        'RPAREN',
        'LBRACE',
        'RBRACE',
    )

    # 2. DICCIONARIO ASOCIATIVO DE PALABRAS RESERVADAS
    reserved = {
        'if': 'KEYWORD_IF',
        'else': 'KEYWORD_ELSE',
        'while': 'KEYWORD_WHILE',
        'int': 'KEYWORD_INT',
        'float': 'KEYWORD_FLOAT'
    }

    # 3. REGLAS SIMPLES DE CADENA (t_TOKEN)
    t_ASSIGN    = r'='
    t_PLUS      = r'\+'
    t_MINUS     = r'-'
    t_TIMES     = r'\*'
    t_DIVIDE    = r'/'
    t_SEMICOLON = r';'
    t_LPAREN    = r'\('
    t_RPAREN    = r'\)'
    t_LBRACE    = r'\{'
    t_RBRACE    = r'\}'

    # Caracteres ignorados (espacios y tabuladores)
    t_ignore = ' \t'

    # 4. REGLAS BASADAS EN FUNCIONES (t_TOKEN(t))
    def t_FLOAT_LITERAL(self, t):
        r'\d+\.\d+([eE][+-]?\d+)?'
        t.value = float(t.value)
        return t

    def t_INTEGER_LITERAL(self, t):
        r'\d+'
        t.value = int(t.value)
        return t

    def t_IDENTIFIER(self, t):
        r'[a-zA-Z_][a-zA-Z0-9_]*'
        # Disambiguación entre palabras reservadas e identificadores de usuario
        t.type = self.reserved.get(t.value, 'IDENTIFIER')
        return t

    def t_newline(self, t):
        r'\n+'
        t.lexer.lineno += len(t.value)

    def t_error(self, t):
        print(f"⚠️  [Error Léxico] Línea {t.lexer.lineno}, Posición {t.lexpos}: Carácter ilegal '{t.value[0]}'")
        t.lexer.skip(1)

    def build(self, **kwargs):
        """Construye el objeto lexer dinámico mediante PLY."""
        self.lexer = lex.lex(object=self, **kwargs)

    def test(self, data: str):
        """Prueba el escáner sobre una cadena fuente imprimiendo los tokens capturados."""
        self.lexer.input(data)
        print(f"{'Línea':<8} | {'Tipo de Token':<18} | {'Lexema / Valor':<20} | {'Posición Abs.':<12}")
        print("-" * 65)
        for tok in self.lexer:
            print(f"{tok.lineno:<8} | {tok.type:<18} | {str(tok.value):<20} | {tok.lexpos:<12}")

# Práctica Base
sample_code = """
int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

#print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
#py_lexer = PythonLexerPLY()
#py_lexer.build()
#py_lexer.test(sample_code)

Overwriting lexer_ply.py


In [21]:
from lexer_ply import PythonLexerPLY
print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
py_lexer = PythonLexerPLY()
py_lexer.build()
py_lexer.test(sample_code)

=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===
Línea    | Tipo de Token      | Lexema / Valor       | Posición Abs.
-----------------------------------------------------------------
2        | KEYWORD_INT        | int                  | 1           
2        | IDENTIFIER         | contador             | 5           
2        | ASSIGN             | =                    | 14          
2        | INTEGER_LITERAL    | 10                   | 16          
2        | SEMICOLON          | ;                    | 18          
3        | KEYWORD_FLOAT      | float                | 20          
3        | IDENTIFIER         | promedio             | 26          
3        | ASSIGN             | =                    | 35          
3        | FLOAT_LITERAL      | 95.5                 | 37          
3        | SEMICOLON          | ;                    | 41          
4        | KEYWORD_WHILE      | while                | 43          
4        | LPAREN             | (                    |

### 🛠️ Desafío 1 (PLY - Python)
**Instrucciones:** Escribe a continuación la extensión de la clase `PythonLexerPLY` (o redefine la clase ampliada `PythonLexerPLYExtended`) para incluir:
1. Soporte para **cadenas literales** entrecomilladas (`"..."`) asignando el token `'STRING_LITERAL'`.
2. Soporte para **comentarios de una línea** (`//...`), configurando el lexer para que sean **ignorados en silencio** sin emitir ningún token.

In [6]:
%%writefile lexer_ply_extended.py

import ply.lex as lex
from lexer_ply import PythonLexerPLY


class PythonLexerPLYExtended(PythonLexerPLY):
    """Extiende PythonLexerPLY con cadenas literales y comentarios de una línea."""

    # 1. Se amplía la lista de tokens (los comentarios NO llevan token, se descartan)
    tokens = PythonLexerPLY.tokens + ('STRING_LITERAL',)

    # 2. Comentarios de una línea: se descartan sin retornar nada
    def t_COMMENT(self, t):
        r'//.*'
        pass  # Al no hacer return, PLY descarta el token en silencio

    # 3. Cadenas entrecomilladas, con soporte para secuencias de escape (\" , \\ , \n ...)
    def t_STRING_LITERAL(self, t):
        r'"([^"\\\n]|\\.)*"'
        t.value = t.value[1:-1]  # Se quitan las comillas delimitadoras
        return t


Writing lexer_ply_extended.py


In [7]:
from lexer_ply_extended import PythonLexerPLYExtended

sample_code_ext = '''
int contador = 10;   // esto es un comentario
string_var = "hola mundo";
url = "http://ejemplo.com";   // el // dentro de la cadena NO es comentario
msg = "dice \\"hola\\"";
// comentario en una línea completa
float x = 3.14;
@
'''

print("=== PRUEBA DEL LEXER EXTENDIDO ===")
lexer_ext = PythonLexerPLYExtended()
lexer_ext.build()
lexer_ext.test(sample_code_ext)

=== PRUEBA DEL LEXER EXTENDIDO ===
Línea    | Tipo de Token      | Lexema / Valor       | Posición Abs.
-----------------------------------------------------------------
2        | KEYWORD_INT        | int                  | 1           
2        | IDENTIFIER         | contador             | 5           
2        | ASSIGN             | =                    | 14          
2        | INTEGER_LITERAL    | 10                   | 16          
2        | SEMICOLON          | ;                    | 18          
3        | IDENTIFIER         | string_var           | 47          
3        | ASSIGN             | =                    | 58          
3        | STRING_LITERAL     | hola mundo           | 60          
3        | SEMICOLON          | ;                    | 72          
4        | IDENTIFIER         | url                  | 74          
4        | ASSIGN             | =                    | 78          
4        | STRING_LITERAL     | http://ejemplo.com   | 80          
4        | SEM

---  
## ☕ PARTE 2: Analizador Léxico en Java utilizando JFlex

### 2.1 Configuración e Instalación del Entorno

In [8]:
import os

print("=== 1. INSTALANDO JDK EN GOOGLE COLAB ===")
ret = os.system("apt-get update -qq && apt-get install -y openjdk-11-jdk-headless > /dev/null")
print("✅ Entorno Java 11 configurado correctamente.\n" if ret == 0 else "❌ Falló la instalación del JDK.\n")

=== 1. INSTALANDO JDK EN GOOGLE COLAB ===
✅ Entorno Java 11 configurado correctamente.



### 2.2 Creación de Archivos Base de Java y Especificación JFlex

In [22]:
# --- PASO 1: DEFINICIÓN DE LA ENUMERACIÓN DE TOKENS (TokenType.java) ---
tokentype_code = """
public enum TokenType {
    // Palabras Reservadas
    KEYWORD_IF, KEYWORD_ELSE, KEYWORD_WHILE, KEYWORD_INT, KEYWORD_FLOAT,

    // Identificadores y Literales
    IDENTIFIER, INTEGER_LITERAL, FLOAT_LITERAL, STRING_LITERAL,

    // Operadores y Delimitadores
    ASSIGN, PLUS, MINUS, TIMES, DIVIDE, SEMICOLON,
    LPAREN, RPAREN, LBRACE, RBRACE,

    // Control de Errores
    ERROR
}
"""

with open("TokenType.java", "w") as f:
    f.write(tokentype_code)

# --- PASO 2: CLASE CONTENEDORA DE TOKENS CON RASTREO (Token.java) ---
token_code = """
public class Token {
    public TokenType type;
    public Object value;
    public int line;
    public int column;

    public Token(TokenType type, Object value, int line, int column) {
        this.type = type;
        this.value = value;
        this.line = line;
        this.column = column;
    }

    @Override
    public String toString() {
        return String.format("%-8d | %-8d | %-18s | %-20s", line, column, type, value);
    }
}
"""

with open("Token.java", "w") as f:
    f.write(token_code)

# --- PASO 3: ARCHIVO DE ESPECIFICACIÓN JFLEX (scanner.jflex) ---
jflex_spec_code = """
import java.io.*;

%%

/* SECCIÓN 1: OPCIONES Y DIRECTIVAS DE CONFIGURACIÓN */
%class Lexer
%public
%unicode
%line
%column
%type Token

%{
    // Métodos auxiliares para construir objetos Token con coordenadas exactas
    private Token symbol(TokenType type) {
        return new Token(type, yytext(), yyline + 1, yycolumn + 1);
    }

    private Token symbol(TokenType type, Object value) {
        return new Token(type, value, yyline + 1, yycolumn + 1);
    }
%}

/* SECCIÓN 2: MACROS Y DEFINICIONES REGULARES */
DIGITO      = [0-9]
LETRA       = [a-zA-Z_]
ID          = {LETRA}({LETRA}|{DIGITO})*
ENTERO      = {DIGITO}+
FLOTANTE    = {DIGITO}+\\.{DIGITO}+([eE][+-]?{DIGITO}+)?
WS          = [ \\t\\r\\n]+

%%

/* SECCIÓN 3: REGLAS LÉXICAS Y ACCIONES EN JAVA */

/* Ignorar espacios en blanco, tabuladores y saltos de línea */
{WS}            { /* Ignorar */ }

/* Palabras Reservadas (prioridad por ubicación superior a ID) */
"if"            { return symbol(TokenType.KEYWORD_IF); }
"else"          { return symbol(TokenType.KEYWORD_ELSE); }
"while"         { return symbol(TokenType.KEYWORD_WHILE); }
"int"           { return symbol(TokenType.KEYWORD_INT); }
"float"         { return symbol(TokenType.KEYWORD_FLOAT); }

/* Literales con conversión de tipo nativo */
{FLOTANTE}      { return symbol(TokenType.FLOAT_LITERAL, Double.parseDouble(yytext())); }
{ENTERO}        { return symbol(TokenType.INTEGER_LITERAL, Integer.parseInt(yytext())); }

/* Identificadores */
{ID}            { return symbol(TokenType.IDENTIFIER, yytext()); }

/* Operadores y Delimitadores */
"="             { return symbol(TokenType.ASSIGN); }
"+"             { return symbol(TokenType.PLUS); }
"-"             { return symbol(TokenType.MINUS); }
"*"             { return symbol(TokenType.TIMES); }
"/"             { return symbol(TokenType.DIVIDE); }
";"             { return symbol(TokenType.SEMICOLON); }
"("             { return symbol(TokenType.LPAREN); }
")"             { return symbol(TokenType.RPAREN); }
"{"             { return symbol(TokenType.LBRACE); }
"}"             { return symbol(TokenType.RBRACE); }

/* Captura de Errores Léxicos (Modo Pánico) */
.               {
                    System.err.printf("⚠️  [Error Léxico] Línea %d, Columna %d: Carácter ilegal '%s'\\n",
                                      yyline + 1, yycolumn + 1, yytext());
                    return symbol(TokenType.ERROR, yytext());
                }
"""

with open("scanner.jflex", "w") as f:
    f.write(jflex_spec_code)

# --- PASO 4: CLASE PRINCIPAL DE PRUEBA (Main.java) ---
main_code = """
import java.io.*;

public class Main {
    public static void main(String[] args) {
        if (args.length == 0) {
            System.out.println("Por favor ingrese la ruta del archivo de prueba.");
            return;
        }

        try {
            FileReader reader = new FileReader(args[0]);
            Lexer scanner = new Lexer(reader);

            System.out.println("=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===");
            System.out.println(String.format("%-8s | %-8s | %-18s | %-20s", "Línea", "Columna", "Tipo de Token", "Lexema / Valor"));
            System.out.println("-".repeat(65));

            Token token;
            while ((token = scanner.yylex()) != null) {
                System.out.println(token);
            }
            System.out.println("\\n=== ESCANEO FINALIZADO EXITOSAMENTE ===");

        } catch (IOException e) {
            System.err.println("Error al leer el archivo: " + e.getMessage());
        }
    }
}
"""

with open("Main.java", "w") as f:
    f.write(main_code)

# --- PASO 5: ARCHIVO DE FUENTE DE PRUEBA (prueba.txt) ---
sample_code = """int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

with open("prueba.txt", "w") as f:
    f.write(sample_code)

print("✅ Archivos Java y especificacion scanner.jflex creados exitosamente.")

✅ Archivos Java y especificacion scanner.jflex creados exitosamente.


### 2.3 Compilación y Ejecución en Colab

In [23]:
print("=== DESCARGANDO JFLEX (JAR con dependencias) ===")
!rm -rf jflex-1.9.1 jflex.tar.gz
!wget -q https://github.com/jflex-de/jflex/releases/download/v1.9.1/jflex-1.9.1.tar.gz -O jflex.tar.gz
!tar -xzf jflex.tar.gz

print("\n=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===")
!java -Xlog:disable -jar jflex-1.9.1/lib/jflex-full-1.9.1.jar scanner.jflex
!javac -J-Xlog:disable TokenType.java Token.java Lexer.java Main.java

print("\n=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===")
!java -Xlog:disable Main prueba.txt

=== DESCARGANDO JFLEX (JAR con dependencias) ===

=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===
Reading "scanner.jflex"
Constructing NFA : 80 states in NFA
Converting NFA to DFA : 
........................................
42 states before minimization, 38 states in minimized DFA
Old file "Lexer.java" saved as "Lexer.java~"
Writing code to "Lexer.java"

=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===
=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===
Línea    | Columna  | Tipo de Token      | Lexema / Valor      
-----------------------------------------------------------------
1        | 1        | KEYWORD_INT        | int                 
1        | 5        | IDENTIFIER         | contador            
1        | 14       | ASSIGN             | =                   
1        | 16       | INTEGER_LITERAL    | 10                  
1        | 18       | SEMICOLON          | ;                   
2        | 1        | KEYWORD_FLOAT      | float               
2        | 7       

### 🛠️ Desafío 2 (JFlex - Java)
**Instrucciones:** Modifica la especificación `scanner.jflex` arriba o vuelve a escribir el archivo desde una celda agregando:
1. La definición de la regla de **cadenas literales entrecomilladas**: `\"[^\"]*\" { return symbol(TokenType.STRING_LITERAL, yytext()); }`
2. La regla para **comentarios de una línea**: `"//".* { /* Ignorar */ }`
Recompila con `!jflex scanner.jflex && javac *.java` y ejecuta sobre un archivo de prueba ampliado.

In [24]:
# Leer la especificación existente
with open("scanner.jflex", "r") as f:
    spec = f.read()

# Reglas nuevas (string RAW: los backslashes llegan intactos al archivo)
new_rules = r'''/* Comentarios de una línea: se ignoran sin emitir token */
"//".*          { /* Ignorar */ }

/* Cadenas literales entrecomilladas */
\"[^\"]*\"      { return symbol(TokenType.STRING_LITERAL, yytext()); }

'''

# Insertarlas justo antes de la sección de operadores
marker = "/* Operadores y Delimitadores */"
assert marker in spec, "No se encontró el marcador en scanner.jflex"
if '"//".*' not in spec:   # evita duplicar si ejecutas la celda dos veces
    spec = spec.replace(marker, new_rules + marker)

with open("scanner.jflex", "w") as f:
    f.write(spec)

print("✅ scanner.jflex actualizado con STRING_LITERAL y comentarios //")

✅ scanner.jflex actualizado con STRING_LITERAL y comentarios //


In [26]:
sample_ext = '''int contador = 10;   // comentario al final de línea
float promedio = 95.5;
// comentario en línea completa
mensaje = "hola mundo";
url = "http://ejemplo.com";   // el // dentro de la cadena NO es comentario
vacia = "";
x = 10 / 2;                   // la división sigue funcionando
@
'''

with open("prueba_ext.txt", "w") as f:
    f.write(sample_ext)

In [31]:
!jflex scanner.jflex && javac *.java && java Main prueba_ext.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Reading "scanner.jflex"
Constructing NFA : 95 states in NFA
Converting NFA to DFA : 
.............................................
47 states before minimization, 42 states in minimized DFA
Old file "Lexer.java" saved as "Lexer.java~"
Writing code to "Lexer.java"
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgrou

In [30]:
!jflex scanner.jflex && javac *.java && java Main prueba_ext.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Reading "scanner.jflex"
Constructing NFA : 95 states in NFA
Converting NFA to DFA : 
.............................................
47 states before minimization, 42 states in minimized DFA
Old file "Lexer.java" saved as "Lexer.java~"
Writing code to "Lexer.java"
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgrou

---  
## 📝 BITÁCORA Y REPORTE DE LABORATORIO

**Nombre del Estudiante:** Juan Francisco Martinez Azuara

**Matrícula / Código:** 2223330175

**Fecha:** 11 de Noviembre de 2026

---
### Sección 1: Verificación de Coincidencia y Resolución de Ambigüedades
1. **Mecanismo de la regla más larga (*Longest Match*):**  
   Al analizar la subcadena `1.2e2`, explica paso a paso por qué el autómata generado no la separa como el entero `1`, el operador `.` y el identificador/entero `2e2`, sino como un único token `FLOAT_LITERAL`.

   ***Respuesta del alumno:***

   JFlex compila todas las expresiones regulares en un único autómata finito determinista (AFD). Ese AFD no se detiene en el primer estado de aceptación: sigue leyendo mientras exista una transición válida y recuerda cuál fue el último estado de aceptación y en qué posición ocurrió. Cuando llega a un estado muerto, retrocede a ese último punto de aceptación y emite el token.
   
   Con 1.2e2 el recorrido es:
   1. Lee 1: el estado es de aceptación para {ENTERO}, pero también es un prefijo válido de {FLOTANTE}, así que continúa.
   2. Lee .: {ENTERO} ya no puede avanzar, pero {FLOTANTE} sí. El estado no es de aceptación (falta la parte decimal).
   3. Lee 2: se llega a un estado de aceptación de {FLOTANTE} (1.2).
   4. Lee e: sigue la rama del exponente, sin aceptación.
   5. Lee 2: nuevo estado de aceptación (1.2e2).
   6. Lee ; o un espacio: no hay transición y se cae al estado muerto. El autómata retrocede al último estado de aceptación, 1.2e2, y emite un único FLOAT_LITERAL.
   
   Entre las candidatas, {ENTERO} cubre 1 carácter, {FLOTANTE} cubre 5 y gana la más larga. Además, la especificación no define ningún token para ., por lo que la separación 1, ., 2e2 ni siquiera sería válida.
   
   En PLY el mecanismo es distinto. No hay AFD: PLY une las expresiones en una alternancia del módulo re, que toma la primera alternativa que coincide, no la más larga. Por eso t_FLOAT_LITERAL debe definirse antes que t_INTEGER_LITERAL; si no, 1.2e2 se partiría en 1 y un error en ..
   >

2. **Disambiguación de Palabras Reservadas vs. Identificadores:**  
   * En **JFlex**, ¿por qué la posición (física) de las reglas de `"if"`, `"while"`, etc., por encima de la regla `{ID}` garantiza que no sean clasificadas como variables?
   * En **PLY**, ¿por qué se utiliza el método `reserved.get(t.value, 'IDENTIFIER')` dentro de la función `t_IDENTIFIER` en lugar de crear una regla independiente `t_KEYWORD_IF = r'if'`?

   ***Respuesta del alumno:***

   JFlex: "if" y {ID} coinciden con la misma longitud sobre el texto if (2 caracteres). Ante un empate de longitud, JFlex elige la regla que aparece primero en el archivo; por eso las palabras reservadas deben ir antes que {ID}. Si {ID} estuviera primero, if se clasificaría como variable. La posición solo desempata: con iffy, {ID} coincide con 4 caracteres y "if" con 2, así que gana {ID} por la regla de la coincidencia más larga y no se parte la palabra.

   PLY: con t_KEYWORD_IF = r'if' habría dos problemas:

   1. PLY procesa primero las reglas de función (como t_IDENTIFIER) y después las de cadena, así que if siempre sería capturado por t_IDENTIFIER y la regla de la palabra reservada nunca se ejecutaría.
   2. Si se evaluara antes, el prefijo if de iffy se reconocería como palabra reservada y quedaría un fy suelto.

   El patrón correcto es reconocer primero el lexema completo con la expresión de identificador y luego consultar el diccionario con reserved.get(t.value, 'IDENTIFIER'). Así se reasigna t.type solo si la palabra completa es reservada, con búsqueda en O(1) y sin una regex por palabra.
   >

---
### Sección 2: Conversión y Tipado de Atributos Léxicos
1. **Conversión de Tipos:**  
   En la regla `t_FLOAT_LITERAL` de PLY se ejecuta `t.value = float(t.value)`, y en JFlex se pasa `Double.parseDouble(yytext())`. ¿Por qué es fundamental que esta conversión ocurra dentro del analizador léxico antes de entregar el token al analizador sintáctico/semántico?  
   
   ***Respuesta del alumno:***

   El lexema es solo texto ("95.5"), mientras que el analizador sintáctico y el semántico necesitan valores con tipo para operar. La conversión debe hacerse en el léxico por estas razones:
   
   1. Operaciones y verificación de tipos: sin conversión, "10" + "5" sería concatenación de cadenas, no suma. El análisis semántico necesita saber que el token es un entero o un flotante.
   2. Separación de responsabilidades: el léxico es la única fase que conoce la forma textual del literal (por ejemplo, la notación científica 1.2e2). Si cada fase posterior reinterpretara el texto, la lógica se duplicaría.
   3. Detección temprana de errores: un literal fuera de rango (por ejemplo, un entero demasiado grande) falla en int() o Integer.parseInt() en el momento de escanearlo, no más adelante en la compilación.
   4. Atributo semántico del token: el token no es solo (tipo, texto), sino (tipo, valor), y ese valor es lo que consumen las fases siguientes.
   >

---
### Sección 3: Análisis Estructural y Manejo de Posiciones
1. **Comparación de Rastreos de Coordenadas:**  
   Explica la diferencia técnica entre cómo rastrea PLY la ubicación de los tokens (utilizando `t.lexer.lineno` y `t.lexpos`) y cómo lo realiza JFlex mediante las directivas `%line` y `%column` con las variables internas `yyline` y `yycolumn`.

   ***Respuesta del alumno:***

   En PLY el rastreo es manual y parcial:
   
   1. lexer.lineno es un contador que PLY no actualiza solo. El programador debe incrementarlo en una regla como t_newline (t.lexer.lineno += len(t.value)); si se omite, todos los tokens salen con la línea 1.
   2. t.lexpos es la posición absoluta (índice desde el inicio de toda la entrada), no la columna. Para obtener la columna hay que calcularla a mano, por ejemplo restando la posición del último salto de línea.
   
   En JFlex el rastreo es automático:

   1. %line y %column hacen que el escáner generado mantenga yyline y yycolumn por sí mismo, reconociendo los terminadores de línea y reiniciando la columna tras cada salto.
   2. Ambas variables son relativas al inicio del lexema reconocido y empiezan en 0, por eso en el código se usa yyline + 1 y yycolumn + 1.

  En resumen: PLY da línea (manual) y posición absoluta, y JFlex entrega línea y columna ya calculadas.
   >

---
### Sección 4: Extensión de Reglas (Evidencia del Desafío)
Pega las líneas de código o expresiones regulares que agregaste en PLY y JFlex para dar soporte a cadenas literales y comentarios de una línea:

**Código/Reglas agregadas en PLY (Python):**

```python
tokens = PythonLexerPLY.tokens + ('STRING_LITERAL',)

def t_COMMENT(self, t):
    r'//.*'
    pass  # sin return: el comentario se descarta

def t_STRING_LITERAL(self, t):
    r'"([^"\\\n]|\\.)*"'
    t.value = t.value[1:-1]
    return t
```

**Reglas agregadas en scanner.jflex (Java):**

```jflex
/* Comentarios de una línea: se ignoran sin emitir token */
"//".*          { /* Ignorar */ }

/* Cadenas literales entrecomilladas */
\"[^\"]*\"      { return symbol(TokenType.STRING_LITERAL, yytext()); }
```

---
### Sección 5: Cuadro Comparativo Arquitectónico

| Criterio de Comparación | PLY (Python) | JFlex (Java) |
| :--- | :--- | :--- |
| **Naturaleza del Generador** | (Interpretado / Reflexivo / Compilado) | (Interpretado / Reflexivo / Compilado) |
| **Definición de Patrones** | Docstrings en funciones Python | Archivo de especificación `.jflex` |
| **Manejo de Palabras Reservadas** |Una sola regla t_IDENTIFIER y consulta al diccionario reserved para reasignar el tipo | Reglas literales ("if", "while"...) colocadas antes de {ID}; el empate de longitud se resuelve por posición|
| **Rastreo de Línea y Columna** | Manual: lineno se actualiza en t_newline; lexpos es posición absoluta y la columna se calcula a mano| Automático con %line y %column (yyline, yycolumn, base 0)|
| **Recuperación de Errores** | Función t_error: informa el error y avanza con t.lexer.skip(1) (el carácter ilegal se descarta)| Regla comodín . al final: informa el error y devuelve un token ERROR (modo pánico)|

---  
**Conclusión General del Experimento:**  
> PLY es rápido de aprender y de modificar: todo vive en una clase Python, las reglas son funciones y se prueba de inmediato sin pasos de compilación, lo que lo hace ideal para prototipos y proyectos académicos. Sus desventajas son que depende del orden de definición de las reglas (el motor re toma la primera coincidencia, no la más larga), que el rastreo de columnas es manual y que el rendimiento es menor al de un escáner compilado.

JFlex genera un AFD compilado que aplica de forma nativa la regla de la coincidencia más larga con desempate por posición, rastrea línea y columna automáticamente y ofrece mayor velocidad y robustez, por lo que es más adecuado para un compilador real. A cambio, exige un flujo de trabajo más pesado (especificación, generación y compilación) y una integración más rígida con Java. En conjunto, PLY favorece la rapidez de desarrollo y JFlex el rendimiento y la corrección en proyectos de mayor escala.